# Results Analysis: Heuristic vs. Q-Learning

This notebook compares the default Manhattan heuristic with the optional tabular Q-learning strategy across different board sizes.

## Reward model

For a terminal sub-game the reward vector is:

$$ R_{cop\_win} = (20, 5), \quad R_{thief\_win} = (5, 10) $$

where the first component is the Cop score and the second component is the Thief score.

## References
- Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press.
- Watkins, C. J. C. H. (1989). Learning from delayed rewards. PhD thesis, University of Cambridge.


In [1]:
import matplotlib.pyplot as plt
import numpy as np

from copthief.agents import HeuristicStrategy, QLearningStrategy
from copthief.agents.training import train_cop_q_learning
from copthief.constants import Outcome, Role
from copthief.services.dialogue import Observation
from copthief.services.game_engine import GameEngine
from copthief.services.scoring import ScoreBook


## Helper: run one sub-game with a given strategy pair

In [2]:
def play_sub_game(grid_size, cop_strategy, thief_strategy, max_moves=25):
    engine = GameEngine(grid_size, max_moves, max_barriers=0)
    while engine.state.outcome == Outcome.ONGOING:
        role = engine.state.turn
        state = engine.state
        if role == Role.COP:
            my_pos, opp_pos = state.cop_pos, state.thief_pos
        else:
            my_pos, opp_pos = state.thief_pos, state.cop_pos
        obs = Observation(
            role=role,
            my_position=my_pos,
            opponent_position=opp_pos,
            barriers=set(state.barriers),
            last_message="",
            move_number=state.move_number,
        )
        action = (cop_strategy if role == Role.COP else thief_strategy).choose_action(obs, "")
        engine.apply_action(role, action)
    return engine.state.outcome, engine.state.move_number


## Experiment: cop-win rate on 5x5

In [3]:
GRID = (5, 5)
N_GAMES = 100

heuristic = HeuristicStrategy(GRID)
trained_ql, _ = train_cop_q_learning(GRID, thief_pos=(4, 4), episodes=300, seed=0)

heuristic_wins = 0
ql_wins = 0
for _ in range(N_GAMES):
    outcome, _ = play_sub_game(GRID, heuristic, heuristic)
    if outcome == Outcome.COP_WIN:
        heuristic_wins += 1
    outcome, _ = play_sub_game(GRID, trained_ql, heuristic)
    if outcome == Outcome.COP_WIN:
        ql_wins += 1

print(f"Heuristic cop win rate: {heuristic_wins / N_GAMES:.2%}")
print(f"Trained Q-learning cop win rate: {ql_wins / N_GAMES:.2%}")


## Board-size comparison

In [4]:
sizes = [(2, 2), (3, 3), (4, 4), (5, 5)]
heuristic_rates = []
ql_rates = []

for size in sizes:
    h = HeuristicStrategy(size)
    ql, _ = train_cop_q_learning(size, thief_pos=(size[0] - 1, size[1] - 1), episodes=200, seed=0)
    h_wins = sum(1 for _ in range(50) if play_sub_game(size, h, h)[0] == Outcome.COP_WIN)
    q_wins = sum(1 for _ in range(50) if play_sub_game(size, ql, h)[0] == Outcome.COP_WIN)
    heuristic_rates.append(h_wins / 50)
    ql_rates.append(q_wins / 50)

x = np.arange(len(sizes))
width = 0.35
plt.figure(figsize=(7, 4))
plt.bar(x - width/2, heuristic_rates, width, label="Heuristic")
plt.bar(x + width/2, ql_rates, width, label="Q-learning")
plt.xticks(x, [f"{r}x{c}" for r, c in sizes])
plt.ylabel("Cop win rate")
plt.xlabel("Board size")
plt.title("Cop win rate by board size and strategy")
plt.legend()
plt.ylim(0, 1)
plt.tight_layout()
plt.show()


## Discussion

The heuristic policy is already strong because it directly minimises/maximises Manhattan distance. The tabular Q-learning agent improves with training but remains limited by its state abstraction (only the agent's own cell). More sophisticated representations would be needed to exceed the heuristic on larger boards.
